In [1]:
import sys

In [ ]:
!{sys.executable} -m pip install xgboost
!{sys.executable} -m pip install torch
!{sys.executable} -m pip install captum
!{sys.executable} -m pip install pygam

In [3]:
import time
import numpy as np
import pandas as pd
import nbimporter

import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [4]:
import synthetic_data as syn

synth= syn.synthetic_data_generator(n=500,df=True)

In [5]:
synth= pd.read_csv('datasets/SVD_Exp_synth_data.csv')

# split synth into features (x) and target (y)
x_syn= synth.loc[:,synth.columns[0:4]]
y_syn= synth.loc[:,synth.columns[4:5]]

x_syn.head()

,core_1,core_2,noise_1,noise_2
0,-1.297434,1.480258,-1.047823,-1.235156
1,-2.255077,1.190942,-0.573914,-3.008057
2,-3.336307,3.258936,-0.030324,-0.361786
3,0.672172,1.928683,-0.456206,-3.051892
4,-1.106266,3.345063,-3.481175,-1.105859


In [6]:
# Normalization
from sklearn.preprocessing import MinMaxScaler

# All dataset is numeric
scaler= MinMaxScaler(feature_range=(0, 1))
norm_x_syn= scaler.fit_transform(np.asarray(x_syn))
norm_x_syn= pd.DataFrame(norm_x_syn, columns=x_syn.columns)

norm_x_syn.head()

,core_1,core_2,noise_1,noise_2
0,0.425977,0.626863,0.368465,0.345838
1,0.359321,0.603430,0.427822,0.123927
2,0.284062,0.770929,0.495906,0.455156
3,0.563072,0.663184,0.442565,0.118440
4,0.439284,0.777905,0.063690,0.362022


In [7]:
# Standarization
from sklearn.preprocessing import StandardScaler

scaler= StandardScaler().fit(np.asarray(x_syn))
# realiza a padronização (média= 0, variância= 1)
stand_x_syn= scaler.transform(np.asarray(x_syn))
stand_x_syn= pd.DataFrame(stand_x_syn, columns=x_syn.columns)

stand_x_syn.head()

,core_1,core_2,noise_1,noise_2
0,-0.103626,0.267269,-0.428075,-0.498028
1,-0.577271,0.116797,-0.225488,-1.265198
2,-1.112041,1.192347,0.006886,-0.120102
3,0.870530,0.500491,-0.175170,-1.284166
4,-0.009076,1.237141,-1.468284,-0.442078


In [8]:
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(norm_x_syn,y_syn,
                                                                                 train_size=0.80,
                                                                                 random_state=1234)

In [9]:
# the instance to be explained (from train dataset)

target_pos= 9

target_inst= train.iloc[target_pos].to_frame().T
target_labl= labels_train.iloc[target_pos].to_frame().T

target_inst

,core_1,core_2,noise_1,noise_2
845,0.46423,0.331611,0.465974,0.769212


# Logistic Regression Classifier

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Define the hyperparameters and their respective values
log_param_grid= {
    'C': [0.001, 0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'random_state': [0],
    'max_iter': [100, 1000, 10000],
    'class_weight': [None, 'balanced']
}

# Create an instance of the Logistic Regression classifier
log_model= LogisticRegression()

# Perform grid search with cross-validation
log_grid_search= GridSearchCV(log_model, log_param_grid, cv=5)
log_grid_search.fit(train, labels_train.values.ravel())

# Retrieve the best hyperparameters and model
log_best_params= log_grid_search.best_params_
log_best_model = log_grid_search.best_estimator_

# Evaluate the best model
log_accuracy= log_best_model.score(test, labels_test.values.ravel())

print("Best Hyperparameters:", log_best_params)
print("Accuracy with Best Model:", log_accuracy)

Best Hyperparameters: {'C': 10, 'class_weight': 'balanced', 'max_iter': 100, 'penalty': 'l1', 'random_state': 0, 'solver': 'liblinear'}
Accuracy with Best Model: 0.97


In [55]:
import joblib

# Create the logistic regressor
log_model= LogisticRegression(C=10, class_weight='balanced', max_iter=100, penalty='l1', 
                              random_state=0, solver='liblinear')

log_model.fit(train, labels_train.values.ravel())

# Save the trained model
joblib.dump(log_model, 'logistic_regression_model.pkl')

acc_log= sklearn.metrics.accuracy_score(labels_test, log_model.predict(test))
acc_log

0.97

In [103]:
# Create a custom PyTorch model that mimics the behavior of a scikit-learn LogisticRegression model

import torch
import torch.nn as nn

# Define the custom PyTorch model
class LogisticRegressionModel(nn.Module):
    
    def __init__(self, coef, intercept):
        super(LogisticRegressionModel, self).__init__()
        self.coef_ = torch.tensor(coef.T)
        self.intercept_ = torch.tensor(intercept)

    def forward(self, x):
        return torch.sigmoid(torch.matmul(x, self.coef_) + self.intercept_)

# Load the scikit-learn model
model_log= joblib.load('logistic_regression_model.pkl')

# Convert the scikit-learn model to a PyTorch model
log_pytorch_model= LogisticRegressionModel(model_log.coef_, model_log.intercept_)

In [104]:
# Evaluate the PyTorch model
test_tensor= torch.from_numpy(np.asarray(test))
log_preds= log_pytorch_model(test_tensor)
log_preds_binary= (log_preds > 0.5).squeeze().numpy()

accuracy= np.mean(log_preds_binary == labels_test.values.ravel().astype(int))

print("Accuracy:", accuracy)

Accuracy: 0.97


In [194]:
# Apply Integrated Gradients to explain predictions

from captum.attr import IntegratedGradients

# Create an instance of the IntegratedGradients class
ig= IntegratedGradients(log_pytorch_model)

# Define the input data for which you want to explain the prediction
input_data= torch.from_numpy(np.asarray(target_inst))

# Compute the attributions using Integrated Gradients
attributions= ig.attribute(input_data)

np.asarray(attributions)

array([[-0.21948943,  0.1455993 ,  0.00635973, -0.00067306]])

In [197]:
# Apply Integrated Gradients to explain predictions

from captum.attr import InputXGradient

# Create an instance of the InputXGradient class
input_x_gradient= InputXGradient(log_pytorch_model)

# Compute the importance scores using Input × Gradient
attributions= input_x_gradient.attribute(input_data)

# Get feature importance scores
np.asarray(attributions.tolist())

array([[-2.67780467e-03,  1.77633377e-03,  7.75897211e-05,
        -8.21147375e-06]])

# K-NN Classifier

In [17]:
from sklearn.neighbors import KNeighborsClassifier

# Define the hyperparameters and their respective values
knn_param_grid = {
    'n_neighbors': [3, 5, 7],
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'metric': ['euclidean', 'manhattan']
}

# Create an instance of the KNN classifier
knn_model= KNeighborsClassifier()

# Perform grid search with cross-validation
knn_grid_search= GridSearchCV(knn_model, knn_param_grid, cv=5)
knn_grid_search.fit(train, labels_train.values.ravel())

# Retrieve the best hyperparameters and model
knn_best_params= knn_grid_search.best_params_
knn_best_model = knn_grid_search.best_estimator_

# Evaluate the best model
knn_accuracy= knn_best_model.score(train, labels_train.values.ravel())

print("Best Hyperparameters:", knn_best_params)
print("Accuracy with Best Model:", knn_accuracy)

Best Hyperparameters: {'algorithm': 'auto', 'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'distance'}
Accuracy with Best Model: 1.0


In [183]:
# Create the KNN classifier 
knn_model= KNeighborsClassifier(algorithm='auto', metric='euclidean', n_neighbors=3, weights='distance')

knn_model.fit(np.asarray(train), labels_train.values.ravel())

# Save the trained model
joblib.dump(knn_model, 'knn_model.pkl')

acc_knn= sklearn.metrics.accuracy_score(labels_test, knn_model.predict(np.asarray(test)))
acc_knn

0.935

In [239]:
# Create a custom PyTorch model that mimics the behavior of a scikit-learn KNeighborsClassifier model

# Define the custom PyTorch model
class KNeighborsClassifierModel(nn.Module):
    def __init__(self, model):
        super(KNeighborsClassifierModel, self).__init__()
        self.model= model

    def forward(self, x):
        _, indices= self.model.kneighbors(x)
        return torch.from_numpy(self.model.classes_).unsqueeze(0), torch.from_numpy(self.model.predict_proba(x))
    
# Load the scikit-learn model
model_knn= joblib.load('knn_model.pkl')

# Convert the scikit-learn model to a PyTorch model
knn_pytorch_model= KNeighborsClassifierModel(model_knn)

In [220]:
# Evaluate the PyTorch model
knn_classes, knn_preds= knn_pytorch_model(test_tensor)

accuracy= np.mean(np.asarray(knn_preds) == labels_test.values.ravel().astype(int))

print("Accuracy:", accuracy)

Accuracy: 0.935


In [241]:
# The scikit-learn KNeighborsClassifier model is not directly compatible with the Integrated Gradients 
# and Gradient x Input methods in Captum. Integrated Gradients requires a model to be differentiable, 
# meaning its parameters can be updated through gradient-based optimization. However, K-nearest neighbors 
#(KNN) models, including KNeighborsClassifier, do not have differentiable parameters.

# Integrated Gradients and Gradient x Input are typically applied to models based on differentiable 
# functions, such as neural networks, where gradients can be computed. KNN models, on the other hand, 
# are based on non-parametric distance metrics and do not have a clear notion of gradients.

# SVM Classifier

In [19]:
from sklearn.svm import SVC

# Define the hyperparameters and their respective values
svm_param_grid= {
    'C': [0.01, 0.1, 1, 10],
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
    'gamma': [0.01, 0.1, 1, 2],
    'class_weight': [None, 'balanced']
}

# Create an instance of the SVM classifier
svm_model= SVC()

# Perform grid search with cross-validation
svm_grid_search= GridSearchCV(svm_model, svm_param_grid, cv=5)
svm_grid_search.fit(train, labels_train.values.ravel())

# Retrieve the best hyperparameters and model
svm_best_params= svm_grid_search.best_params_
svm_best_model = svm_grid_search.best_estimator_

# Evaluate the best model
svm_accuracy= svm_best_model.score(train, labels_train.values.ravel())

print("Best Hyperparameters:", svm_best_params)
print("Accuracy with Best Model:", svm_accuracy)

Best Hyperparameters: {'C': 10, 'class_weight': None, 'gamma': 2, 'kernel': 'rbf'}
Accuracy with Best Model: 0.97


In [73]:
# Create the SVM classifier 
svm_model= SVC(C=10, class_weight=None, gamma=2, kernel='rbf')

svm_model.fit(train, labels_train.values.ravel())

acc_svm= sklearn.metrics.accuracy_score(labels_test, svm_model.predict(test))
acc_svm

0.975

In [ ]:
# The scikit-learn SVC (Support Vector Classifier) model is not directly compatible with the Integrated 
# Gradients method in Captum. Integrated Gradients requires a model to be differentiable, meaning its 
# parameters can be updated through gradient-based optimization. However, the SVC model in scikit-learn, 
# which is based on Support Vector Machines, does not have differentiable parameters.

# Integrated Gradients is typically applied to models based on differentiable functions, such as neural 
# networks, where gradients can be computed. Support Vector Machines, including the SVC model, involve 
# solving an optimization problem to find the hyperplane that maximally separates the data points. This 
# optimization process does not provide direct access to gradients for attribution computations.

# Random Forest Classifier

In [22]:
from sklearn.ensemble import RandomForestClassifier

# Define the hyperparameters and their respective values
rfc_param_grid= {
    'n_estimators': [100, 200, 500],
    'max_depth': [3, 5, 6, 7, 9],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2, 4],
    'random_state': [0],
    'max_features': ['sqrt', 'log2', None],
    'class_weight': [None, 'balanced']
}

# Create an instance of the Random Forest Classifier
rfc_model= RandomForestClassifier()

# Perform grid search with cross-validation
rfc_grid_search= GridSearchCV(rfc_model, rfc_param_grid, cv=5)
rfc_grid_search.fit(train, labels_train.values.ravel())

# Retrieve the best hyperparameters and model
rfc_best_params= rfc_grid_search.best_params_
rfc_best_model = rfc_grid_search.best_estimator_

# Evaluate the best model
rfc_accuracy= rfc_best_model.score(train, labels_train.values.ravel())

print("Best Hyperparameters:", rfc_best_params)
print("Accuracy with Best Model:", rfc_accuracy)

Best Hyperparameters: {'class_weight': None, 'max_depth': 7, 'max_features': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 500, 'random_state': 0}
Accuracy with Best Model: 0.98625


In [250]:
# Create the RF classifier 
rfc_model= RandomForestClassifier(max_depth=7, 
                                  min_samples_leaf=2, min_samples_split=2, n_estimators=500, 
                                  random_state=0)

rfc_model.fit(np.asarray(train), labels_train.values.ravel())

# Save the trained model
joblib.dump(rfc_model, 'random_forest_model.pkl')

acc_rfc= sklearn.metrics.accuracy_score(labels_test, rfc_model.predict(np.asarray(test)))
acc_rfc

0.96

In [253]:
# Define the custom PyTorch model
class RandomForestClassifierModel(nn.Module):
    def __init__(self, model):
        super(RandomForestClassifierModel, self).__init__()
        self.model= model

    def forward(self, x):
        # Perform inference using the RandomForestClassifier model
        probabilities= self.model.predict_proba(x)
        #probabilities= self.model.predict(x)
        return torch.from_numpy(probabilities)

# Load the scikit-learn model
model_rf= joblib.load('random_forest_model.pkl')

# Convert the scikit-learn model to a PyTorch model
rfc_pytorch_model= RandomForestClassifierModel(model_rf)

In [252]:
# Evaluate the PyTorch model
test_tensor= torch.from_numpy(np.asarray(test))
rfc_preds= rfc_pytorch_model(test_tensor)

accuracy= np.mean(np.asarray(rfc_preds) == labels_test.values.ravel().astype(int))

print("Accuracy:", accuracy)

Accuracy: 0.96


In [264]:
# The scikit-learn RandomForestClassifier model is not directly compatible with the Integrated Gradients 
# method in Captum. Integrated Gradients requires a model to be differentiable, meaning its parameters 
# can be updated through gradient-based optimization. However, the RandomForestClassifier model in 
# scikit-learn is an ensemble model that combines multiple decision trees, and individual decision trees 
# within the ensemble are not differentiable.

# Integrated Gradients is typically applied to models based on differentiable functions, such as neural 
# networks, where gradients can be computed. Random Forest models, on the other hand, are based on an 
# ensemble of decision trees, and each decision tree is trained independently using different splitting rules.

# If you want to use the Integrated Gradients method with a RandomForestClassifier model, you would need to 
# convert the RandomForestClassifier into a differentiable model representation. One possible approach is to 
# create a surrogate model, such as a neural network, that approximates the behavior of the 
# RandomForestClassifier. This approximation would allow you to compute gradients and apply the 
# Integrated Gradients method.

# Converting a scikit-learn RandomForestClassifier model into a differentiable model representation for 
# use with the Integrated Gradients method requires approximating the behavior of the random forest. 
# One approach is to create a surrogate model, such as a neural network, that approximates the 
# predictions of the random forest.

# XGBoost RF Classifier

In [21]:
import xgboost as xgb

# Define the hyperparameters and their respective values
xgb_param_grid= {
    'max_depth': [3, 5, 6, 7],
    'learning_rate': [0.01, 0.1],
    'n_estimators': [100, 200, 500],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.6, 0.8],
    'gamma': [0.1, 0.2, 0.5, 1],
    'objective': ['binary:logistic'],
    'eval_metric': ['logloss']
}

# Create an instance of the XGBoost classifier
xgb_model= xgb.XGBRFClassifier()

#eval_set= [(train, labels_train), (test, labels_test)]

#xgb_model.fit(train, labels_train, eval_set=eval_set, verbose=False)

#sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

# Perform grid search with cross-validation
xgb_grid_search= GridSearchCV(xgb_model, xgb_param_grid, cv=5)
xgb_grid_search.fit(train, labels_train.values.ravel())

# Retrieve the best hyperparameters and model
xgb_best_params= xgb_grid_search.best_params_
xgb_best_model = xgb_grid_search.best_estimator_

# Evaluate the best model
xgb_accuracy= xgb_best_model.score(train, labels_train.values.ravel())

print("Best Hyperparameters:", xgb_best_params)
print("Accuracy with Best Model:", xgb_accuracy)

Best Hyperparameters: {'colsample_bytree': 0.8, 'eval_metric': 'logloss', 'gamma': 0.1, 'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 500, 'objective': 'binary:logistic', 'subsample': 0.9}
Accuracy with Best Model: 0.9825


In [261]:
import pickle

# Create the XGBoost classifier 
xgb_model= xgb.XGBRFClassifier(learning_rate= 0.01, n_estimators= 500, max_depth= 6,
                               gamma= 0.1, subsample= 0.9,
                               objective= 'binary:logistic',
                               eval_metric='logloss')

xgb_model.fit(train, labels_train.values.ravel())

# Save the trained model
with open('xgbrf_model.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)

acc_xgb= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))
acc_xgb

0.96

In [262]:
# Define the custom PyTorch model
class XGBRFClassifierModel(nn.Module):
    def __init__(self, model):
        super(XGBRFClassifierModel, self).__init__()
        self.model= model

    def forward(self, x):
        logits= self.model.predict(xgb.DMatrix(x))
        probabilities= torch.softmax(torch.from_numpy(logits), dim=1)
        return probabilities

# Load the XGBoost model
with open('xgbrf_model.pkl', 'rb') as f:
    model_xgb= pickle.load(f)

# Convert the XGBoost model to a PyTorch model
xgb_pytorch_model= XGBRFClassifierModel(model_xgb)

# Neural Networks -- 1, 2, and 3 hyden layers

In [25]:
from sklearn.neural_network import MLPClassifier

# Define the hyperparameters and their respective values
nn1_param_grid= {
    'hidden_layer_sizes': [(64,), (128,), (256,)],
    'activation': ['logistic', 'tanh', 'relu'],
    'solver': ['sgd', 'adam'],
    'learning_rate_init': [0.001, 0.01, 0.1],
    'random_state': [0],
    'max_iter': [500, 1000],
    'alpha': [0.001, 0.01]
}

# Create an instance of the MLPClassifier
nn1_model= MLPClassifier()

# Perform grid search with cross-validation
nn1_grid_search= GridSearchCV(nn1_model, nn1_param_grid, cv=5, scoring='accuracy')
nn1_grid_search.fit(train, labels_train.values.ravel())

# Retrieve the best hyperparameters and model
nn1_best_params= nn1_grid_search.best_params_
nn1_best_model = nn1_grid_search.best_estimator_

# Evaluate the best model
nn1_accuracy= nn1_best_model.score(train, labels_train.values.ravel())

print("Best Hyperparameters:", nn1_best_params)
print("Accuracy with Best Model:", nn1_accuracy)

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anacond

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anacon

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anacon

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anac

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaco

...

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/ana

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/ana

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anacon

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anacon

Best Hyperparameters: {'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (256,), 'learning_rate_init': 0.001, 'max_iter': 500, 'random_state': 0, 'solver': 'adam'}
Accuracy with Best Model: 0.9675


In [78]:
# Create the 1-hyden layer Neural Net classifier 
nn1_model= MLPClassifier(activation='relu', alpha=0.01, hidden_layer_sizes=(256,), 
                         learning_rate_init=0.001, max_iter=500, random_state=0, solver='adam')

nn1_model.fit(train, labels_train.values.ravel())

acc_nn1= sklearn.metrics.accuracy_score(labels_test, nn1_model.predict(test))
acc_nn1

0.975

In [26]:
# Define the hyperparameters and their respective values
nn2_param_grid= {
    'hidden_layer_sizes': [(64,64), (128,128), (256,256), (512,512)],
    'activation': ['logistic', 'tanh', 'relu'],
    'solver': ['sgd', 'adam'],
    'learning_rate_init': [0.001, 0.01, 0.1],
    'random_state': [0],
    'max_iter': [500, 1000],
    'alpha': [0.0001, 0.001, 0.01]
}

# Create an instance of the MLPClassifier
nn2_model= MLPClassifier()

# Perform grid search with cross-validation
nn2_grid_search= GridSearchCV(nn2_model, nn2_param_grid, cv=5, scoring='accuracy')
nn2_grid_search.fit(train, labels_train.values.ravel())

# Retrieve the best hyperparameters and model
nn2_best_params= nn2_grid_search.best_params_
nn2_best_model = nn2_grid_search.best_estimator_

# Evaluate the best model
nn2_accuracy= nn2_best_model.score(train, labels_train.values.ravel())

print("Best Hyperparameters:", nn2_best_params)
print("Accuracy with Best Model:", nn2_accuracy)

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anacond

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anacond

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anacond

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/ana

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anacond

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/ana

Best Hyperparameters: {'activation': 'relu', 'alpha': 0.001, 'hidden_layer_sizes': (64, 64), 'learning_rate_init': 0.001, 'max_iter': 500, 'random_state': 0, 'solver': 'adam'}
Accuracy with Best Model: 0.96875


In [79]:
# Create the 2-hyden layers Neural Net classifier 
nn2_model= MLPClassifier(activation='relu', alpha=0.001, hidden_layer_sizes=(64, 64), 
                         learning_rate_init=0.001, max_iter=500, random_state=0, solver='adam')

nn2_model.fit(train, labels_train.values.ravel())

acc_nn2= sklearn.metrics.accuracy_score(labels_test, nn2_model.predict(test))
acc_nn2

0.97

In [27]:
# Define the hyperparameters and their respective values
nn3_param_grid= {
    'hidden_layer_sizes': [(64,64,64), (128,128,128), (256,256,256), (512,512,512)],
    'activation': ['logistic', 'tanh', 'relu'],
    'solver': ['sgd', 'adam'],
    'learning_rate_init': [0.001, 0.01, 0.1],
    'random_state': [0],
    'max_iter': [500, 1000],
    'alpha': [0.0001, 0.001, 0.01]
}

# Create an instance of the MLPClassifier
nn3_model= MLPClassifier()

# Perform grid search with cross-validation
nn3_grid_search= GridSearchCV(nn3_model, nn3_param_grid, cv=5, scoring='accuracy')
nn3_grid_search.fit(train, labels_train.values.ravel())

# Retrieve the best hyperparameters and model
nn3_best_params= nn3_grid_search.best_params_
nn3_best_model = nn3_grid_search.best_estimator_

# Evaluate the best model
nn3_accuracy= nn3_best_model.score(train, labels_train.values.ravel())

print("Best Hyperparameters:", nn3_best_params)
print("Accuracy with Best Model:", nn3_accuracy)

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anacond

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anacond

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anacond

/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anaconda3/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:684: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/evortigosa/anacond

Best Hyperparameters: {'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (64, 64, 64), 'learning_rate_init': 0.01, 'max_iter': 500, 'random_state': 0, 'solver': 'sgd'}
Accuracy with Best Model: 0.9675


In [80]:
# Create the 2-hyden layers Neural Net classifier 
nn3_model= MLPClassifier(activation='relu', alpha=0.0001, hidden_layer_sizes=(64, 64, 64), 
                         learning_rate_init=0.01, max_iter=500, random_state=0, solver='sgd')

nn3_model.fit(train, labels_train.values.ravel())

acc_nn3= sklearn.metrics.accuracy_score(labels_test, nn3_model.predict(test))
acc_nn3

0.975

In [22]:
def lp_norm_dif(v1, v2, p_norm=2, eps=1e-6, norm:bool=True):
    
    # arrays can be flattened, so long as ordering is preserved
    flat_dif= np.asarray(v1).flatten() - np.asarray(v2).flatten()
    
    if (norm==True): print('dif before div', flat_dif)
    
    if (norm==True):
        v1_aux= np.asarray(v1).flatten()
        v1_aux= np.clip(v1_aux, eps, None)
        
        if (norm==True): print('dif after clip', v1_aux)
                    
        flat_dif= np.divide(flat_dif, v1_aux, where=v1_aux!= 0)
        
    if (norm==True): print('dif afterr div', flat_dif)

    return np.linalg.norm(flat_dif, ord=p_norm)

In [11]:
a= np.asarray([ 3.0089e-05,  4.1239e-03, -2.3394e-18, -1.4968e-18])
b= np.asarray([-3.0089e-05,  1.2576e-04,  2.6217e-19,  1.1082e-18])

In [12]:
lp_norm_dif(a, b, p_norm=2, eps=1e-6, norm=True)

dif before div [ 6.01780e-05  3.99814e-03 -2.60157e-18 -2.60500e-18]
dif after clip [3.0089e-05 4.1239e-03 1.0000e-06 1.0000e-06]
dif afterr div [ 2.00000000e+00  9.69504595e-01 -2.60157000e-12 -2.60500000e-12]


2.2225973904523526

In [17]:
def lp_norm_dif2(v1, v2, p_norm=2, eps=1e-5, norm:bool=True):
    
    # arrays can be flattened, so long as ordering is preserved
    v1_flat= np.asarray(v1).flatten()
    v2_flat= np.asarray(v2).flatten()
    
    dif_flat= (v1_flat - v2_flat)
    
    if (norm==True): print('dif before div', dif_flat)
    
    if (norm==True):
        #v1_flat= np.clip(v1_flat, eps, None)
        
        elements= len(v1_flat)

        for i in range(elements):
            if (v1_flat[i]< 0 and np.abs(v1_flat[i])< eps):
                v1_flat[i]= -eps
            elif (v1_flat[i]> 0 and v1_flat[i]< eps):
                v1_flat[i]= eps
                
        if (norm==True): print('dif after clip', v1_flat)
         
        dif_flat= 1 - np.divide(v2_flat, v1_flat, out=np.zeros_like(v1_flat), where=v1_flat!=0)
        #dif_flat= np.divide(dif_flat, v1_flat, out=np.zeros_like(v1_flat), where=v1_flat!=0)
        
    if (norm==True): print('dif afterr div', dif_flat)

    return np.linalg.norm(dif_flat, ord=p_norm)

In [16]:
# sollution -- (v1 - v2)/v1 

lp_norm_dif2(a, b, p_norm=2, eps=1e-6, norm=True)

dif before div [ 6.01780e-05  3.99814e-03 -2.60157e-18 -2.60500e-18]
dif after clip [ 3.0089e-05  4.1239e-03 -1.0000e-06 -1.0000e-06]
dif afterr div [2.00000000e+00 9.69504595e-01 2.60157000e-12 2.60500000e-12]


2.2225973904523526

In [18]:
# sollution -- 1 - v2/v1 

lp_norm_dif2(a, b, p_norm=2, eps=1e-6, norm=True)

dif before div [ 6.01780e-05  3.99814e-03 -2.60157e-18 -2.60500e-18]
dif after clip [ 3.0089e-05  4.1239e-03 -1.0000e-06 -1.0000e-06]
dif afterr div [2.        0.9695046 1.        1.       ]


2.6343764271736774

In [21]:
# no clipping

lp_norm_dif(a, b, p_norm=2, eps=1e-6, norm=True)

dif before div [ 6.01780e-05  3.99814e-03 -2.60157e-18 -2.60500e-18]
dif after clip [ 3.0089e-05  4.1239e-03 -2.3394e-18 -1.4968e-18]
dif afterr div [2.         0.9695046  1.1120672  1.74037948]


3.0340654790715176